# Credit Card Fraud Detection (Level Up from Churn)

This project levels up our classification skills by introducing:
- **Extreme Class Imbalance**: Realistic scenarios where fraud is < 1% of transactions.
- **Precision-Recall Curves**: Why ROC-AUC can be misleading for imbalanced data.
- **Model Comparison**: Logistic Regression vs. Random Forest.
- **Threshold Tuning**: Moving the decision boundary to optimize for business value.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    roc_curve,
    roc_auc_score,
    precision_recall_curve,
    auc
)

## 1. Generate Synthetic Imbalanced Dataset
Instead of a 25% churn rate, we will generate a dataset where only ~1% of the transactions are fraudulent.


In [ ]:
from sklearn.datasets import make_classification

# Generate highly imbalanced synthetic data
X, y = make_classification(
    n_samples=10000, 
    n_features=10, 
    n_informative=4, 
    n_redundant=2, 
    weights=[0.99, 0.01], # 99% normal, 1% fraud
    random_state=42
)

feature_names = [f"V{i}" for i in range(1, 11)]
df = pd.DataFrame(X, columns=feature_names)
df['Class'] = y

print("Class Distribution:")
print(df['Class'].value_counts(normalize=True))
print("-" * 50)


## 2. Prepare Data
Split the data into training and testing sets, and scale the features.


In [ ]:
X = df[feature_names]
y = df['Class']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

## 3. Train Models (Model Comparison)
We will train a simple Logistic Regression and a more complex Random Forest Classifier.


In [ ]:
# Logistic Regression
lr_model = LogisticRegression(class_weight='balanced', random_state=42)
lr_model.fit(X_train, y_train)

# Random Forest
rf_model = RandomForestClassifier(class_weight='balanced', random_state=42, n_estimators=100)
rf_model.fit(X_train, y_train)

## 4. Evaluation and ROC-AUC
Let's compare their ROC-AUC and standard Classification Reports.


In [ ]:
def evaluate_model(model, name):
    y_pred = model.predict(X_test)
    y_probs = model.predict_proba(X_test)[:, 1]
    
    print(f"--- {name} ---")
    print(f"ROC-AUC: {roc_auc_score(y_test, y_probs):.4f}")
    print("Classification Report:")
    print(classification_report(y_test, y_pred))
    print("-" * 50)

evaluate_model(lr_model, "Logistic Regression")
evaluate_model(rf_model, "Random Forest")


## 5. Precision-Recall Curve (The Right Metric for Imbalance)
ROC-AUC can be overly optimistic for highly imbalanced datasets. Precision-Recall (PR) curves are much more informative here.


In [ ]:
# Calculate PR curves
lr_prec, lr_rec, _ = precision_recall_curve(y_test, lr_model.predict_proba(X_test)[:, 1])
rf_prec, rf_rec, _ = precision_recall_curve(y_test, rf_model.predict_proba(X_test)[:, 1])

plt.figure(figsize=(8, 6))
plt.plot(lr_rec, lr_prec, label=f"Logistic Regression (PR AUC = {auc(lr_rec, lr_prec):.2f})")
plt.plot(rf_rec, rf_prec, label=f"Random Forest (PR AUC = {auc(rf_rec, rf_prec):.2f})")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve Comparison")
plt.legend()
plt.show()

## 6. Threshold Tuning
By default, `.predict()` uses a 0.5 threshold. For fraud, we might want to catch more fraud even if it means more false alarms (increase recall). We can tune the threshold to optimize for our business needs.


In [ ]:
y_probs_rf = rf_model.predict_proba(X_test)[:, 1]

# Find the best threshold that optimizes F1-score
precisions, recalls, thresholds = precision_recall_curve(y_test, y_probs_rf)
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-10)
best_ix = np.argmax(f1_scores)
best_threshold = thresholds[best_ix]

print(f"Default Threshold: 0.50 | Default F1: {f1_score(y_test, rf_model.predict(X_test)):.4f}")
print(f"Optimal Threshold: {best_threshold:.4f} | Optimal F1: {f1_scores[best_ix]:.4f}")

# Apply optimal threshold
y_pred_tuned = (y_probs_rf >= best_threshold).astype(int)

print("\nClassification Report with Optimal Threshold:")
print(classification_report(y_test, y_pred_tuned))

# Confusion Matrix with Tuned Threshold
cm = confusion_matrix(y_test, y_pred_tuned)
plt.figure()
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title(f"Random Forest Confusion Matrix (Threshold = {best_threshold:.2f})")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()